# Ensembl Genomes — Multi-Kingdom Genome Annotation

[Ensembl Genomes](https://ensemblgenomes.org/) is an ELIXIR Core Data Resource that extends the Ensembl project beyond vertebrates to cover the full breadth of life. It provides genome sequence, gene annotation, comparative genomics, and variation data for thousands of non-vertebrate species through five specialised sub-portals.

| Sub-portal | Kingdom | Example Organism | Approx. Genomes |
|---|---|---|---|
| EnsemblBacteria | Bacteria | *Escherichia coli* K-12 MG1655 | ~50,000 |
| EnsemblFungi | Fungi | *Saccharomyces cerevisiae* S288C | ~1,000 |
| EnsemblMetazoa | Metazoa (non-vertebrate animals) | *Drosophila melanogaster* | ~130 |
| EnsemblPlants | Viridiplantae | *Arabidopsis thaliana* Col-0 | ~120 |
| EnsemblProtists | Protists | *Plasmodium falciparum* 3D7 | ~250 |

**Reference:** Cunningham F. *et al.* (2022). Ensembl 2022. *Nucleic Acids Research*, 50(D1), D988–D995. https://doi.org/10.1093/nar/gkab1049

In [ ]:
import requests
import time
import re
import json
from pathlib import Path

import polars as pl
import pandas as pd

# TODO

* [x] **Ingest data**
    * [x] Connect to Ensembl REST API and confirm access
    * [x] Fetch species list for EnsemblPlants division, cache to `data/ensembl_plants_species.json`
    * [x] Fetch species list for EnsemblBacteria division (sample), cache to `data/ensembl_bacteria_species.json`
    * [x] Look up a well-known plant gene in *Arabidopsis thaliana* (AT1G01010), print gene info, biotype, coordinates
    * [x] Build a cross-division summary DataFrame (division, species_count)
    * [x] Fetch gene counts for model organisms across divisions (*A. thaliana*, *S. cerevisiae*, *D. melanogaster*)
    * [x] Fetch genome assembly info for representative species across all portals
    * [x] Parse all data into Polars DataFrames; print shape, dtypes, head
* [ ] **Explore and clean**
    * [ ] Gene count distributions across species
    * [ ] Biotype breakdown (protein-coding, lncRNA, pseudogene, etc.)
    * [ ] Chromosome/scaffold statistics (length distributions, N50)
* [ ] **Comparative genomics**
    * [ ] Orthologue counts across kingdoms
    * [ ] Synteny analysis introduction
* [ ] **Functional annotation**
    * [ ] GO term coverage per species
    * [ ] Pathway annotations across species
* [ ] **Visualization**
    * [ ] Genome size vs gene count scatter plot
    * [ ] Biotype pie charts per species
    * [ ] Cross-kingdom ortholog heatmap
* [ ] **Statistical analysis**
    * [ ] Gene density distributions
    * [ ] Intron/exon statistics
    * [ ] Annotation completeness assessment

## 1. Ingest Data

### 1.1 Connect to Ensembl REST API

In [ ]:
ENSEMBL_BASE = "https://rest.ensembl.org"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Standard headers required by the Ensembl REST API
HEADERS = {"Content-Type": "application/json"}


def eg_get(endpoint: str, params: dict | None = None) -> dict | list:
    """Send a GET request to the Ensembl REST API.

    Parameters
    ----------
    endpoint : str
        API endpoint path, e.g. ``"/info/ping"``. Will be appended to
        ``ENSEMBL_BASE``.
    params : dict or None, optional
        Query parameters to include in the request. Defaults to None.

    Returns
    -------
    dict or list
        Parsed JSON response from the API.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.
    """
    url = f"{ENSEMBL_BASE}{endpoint}"
    response = requests.get(url, headers=HEADERS, params=params, timeout=60)
    response.raise_for_status()
    time.sleep(0.1)   # polite delay; Ensembl allows up to 15 req/s
    return response.json()


# --- connectivity check ---
ping = eg_get("/info/ping")
print("Ping response:", ping)

software = eg_get("/info/software")
print("Ensembl version:", software.get("release"))

### 1.2 Fetch Genome Info for Representative Species

In [ ]:
# Representative species mapped to their Ensembl sub-portal
SPECIES = {
    "arabidopsis_thaliana":                        "EnsemblPlants",
    "saccharomyces_cerevisiae":                    "EnsemblFungi",
    "drosophila_melanogaster":                     "EnsemblMetazoa",
    "escherichia_coli_str_k_12_substr_mg1655":     "EnsemblBacteria",
    "homo_sapiens":                                "Ensembl",
}

ASSEMBLY_FILE = DATA_DIR / "ensembl_genomes_assembly_info.json"

if ASSEMBLY_FILE.exists():
    print(f"Loading cached assembly info from {ASSEMBLY_FILE}")
    with open(ASSEMBLY_FILE) as fh:
        assembly_data = json.load(fh)
else:
    assembly_data = {}
    for species, portal in SPECIES.items():
        print(f"Fetching assembly info for {species} ({portal}) ...")
        info = eg_get(f"/info/assembly/{species}")
        info["_portal"] = portal   # attach portal label for later use
        assembly_data[species] = info

    with open(ASSEMBLY_FILE, "w") as fh:
        json.dump(assembly_data, fh, indent=2)
    print(f"Saved assembly info to {ASSEMBLY_FILE}")

print()
for species, info in assembly_data.items():
    name      = info.get("assembly_name", "N/A")
    total_len = info.get("base_pairs", 0)
    top_level = len(info.get("top_level_region", []))
    print(f"{species:55s}  assembly={name}  total_bp={total_len:,}  top_level_regions={top_level}")

### 1.3 Download Gene Annotations for Arabidopsis thaliana

In [ ]:
ATHAL_CHR1_FILE = DATA_DIR / "arabidopsis_chr1_genes.json"

# Arabidopsis thaliana chromosome 1 is 30,427,671 bp (TAIR10 assembly)
ATHAL_CHR1_END = 30_427_671

if ATHAL_CHR1_FILE.exists():
    print(f"Loading cached Arabidopsis chr1 genes from {ATHAL_CHR1_FILE}")
    with open(ATHAL_CHR1_FILE) as fh:
        athal_genes = json.load(fh)
else:
    print("Fetching Arabidopsis thaliana chromosome 1 genes ...")
    # /overlap/region returns all features overlapping a genomic window
    athal_genes = eg_get(
        f"/overlap/region/arabidopsis_thaliana/1:1-{ATHAL_CHR1_END}",
        params={"feature": "gene", "content-type": "application/json"},
    )
    with open(ATHAL_CHR1_FILE, "w") as fh:
        json.dump(athal_genes, fh, indent=2)
    print(f"Saved to {ATHAL_CHR1_FILE}")

print(f"Number of genes fetched on chr1: {len(athal_genes)}")

### 1.4 Parse into DataFrames

In [ ]:
def to_snake(name: str) -> str:
    """Convert a CamelCase or mixed string to snake_case.

    Parameters
    ----------
    name : str
        Input column name in any case convention.

    Returns
    -------
    str
        Normalised snake_case column name.
    """
    # Insert underscore before uppercase letters that follow lowercase letters
    s = re.sub(r"(?<=[a-z0-9])([A-Z])", r"_\1", name)
    return s.lower().replace(" ", "_").replace("-", "_")


# ---------------------------------------------------------------------------
# Assembly info DataFrame
# ---------------------------------------------------------------------------
assembly_rows = []
for species, info in assembly_data.items():
    top_level = info.get("top_level_region", [])
    # Separate chromosomes (coord_system == "chromosome") from scaffolds
    chromosomes = [r for r in top_level if r.get("coord_system") == "chromosome"]
    scaffolds   = [r for r in top_level if r.get("coord_system") != "chromosome"]

    assembly_rows.append({
        "species":          species,
        "portal":           info.get("_portal", ""),
        "assembly_name":    info.get("assembly_name", ""),
        "total_length_bp":  info.get("base_pairs", None),
        "num_chromosomes":  len(chromosomes),
        "num_scaffolds":    len(scaffolds),
    })

df_assembly = pl.DataFrame(assembly_rows)

print("Assembly DataFrame — shape:", df_assembly.shape)
print(df_assembly.head(5))


# ---------------------------------------------------------------------------
# Arabidopsis chr1 genes DataFrame
# ---------------------------------------------------------------------------
gene_rows = []
for gene in athal_genes:
    gene_rows.append({
        "gene_id":     gene.get("id", ""),
        "gene_name":   gene.get("external_name", ""),
        "biotype":     gene.get("biotype", ""),
        "start":       gene.get("start", None),
        "end":         gene.get("end", None),
        "strand":      gene.get("strand", None),
        "description": gene.get("description", ""),
    })

df_genes = pl.DataFrame(gene_rows)

print("\nArabidopsis chr1 genes DataFrame — shape:", df_genes.shape)
print(df_genes.head(5))

## Column Descriptions

### `df_assembly` — one row per representative species

| Column | Type | Description |
|---|---|---|
| `species` | str | Ensembl species identifier (lowercase, underscores) |
| `portal` | str | Ensembl sub-portal the species belongs to (e.g. `EnsemblPlants`) |
| `assembly_name` | str | Official genome assembly name (e.g. `TAIR10`, `GRCh38`) |
| `total_length_bp` | int | Total number of base pairs in the assembly (sum over all sequences) |
| `num_chromosomes` | int | Count of top-level regions with `coord_system == "chromosome"` |
| `num_scaffolds` | int | Count of top-level regions that are not chromosomes (scaffolds, contigs, etc.) |

### `df_genes` — one row per gene on Arabidopsis thaliana chromosome 1

| Column | Type | Description |
|---|---|---|
| `gene_id` | str | Stable Ensembl gene identifier (e.g. `AT1G01010`) |
| `gene_name` | str | Human-readable gene symbol or name if available |
| `biotype` | str | Ensembl biotype classification (e.g. `protein_coding`, `lncRNA`, `pseudogene`) |
| `start` | int | 1-based genomic start coordinate on chromosome 1 |
| `end` | int | 1-based genomic end coordinate on chromosome 1 |
| `strand` | int | Strand of the feature: `1` = forward, `-1` = reverse |
| `description` | str | Free-text functional description from Ensembl/UniProt, may be empty |

### 1.5 Fetch EnsemblPlants Species List

In [ ]:
PLANTS_CACHE = DATA_DIR / "ensembl_plants_species.json"


def fetch_division_species(division: str, cache_path: Path) -> list[dict]:
    """Fetch the full species list for one Ensembl Genomes division.

    Parameters
    ----------
    division : str
        Ensembl Genomes division name, e.g. ``"EnsemblPlants"``.
    cache_path : Path
        File to read from / write to so subsequent runs are instant.

    Returns
    -------
    list[dict]
        One dict per genome entry returned by the API.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    print(f"Fetching species list for {division} ...")
    # /info/genomes/division/<name> returns metadata for every genome in a division
    data = eg_get(f"/info/genomes/division/{division}")
    cache_path.write_text(json.dumps(data, indent=2))
    print(f"  {len(data)} genomes — saved to {cache_path}")
    return data


plants_raw = fetch_division_species("EnsemblPlants", PLANTS_CACHE)
print(f"\nEnsemblPlants genomes: {len(plants_raw)}")
print("Sample keys:", list(plants_raw[0].keys())[:10])

In [ ]:
def parse_division_species(raw: list[dict]) -> pl.DataFrame:
    """Parse raw division species records into a tidy Polars DataFrame.

    Parameters
    ----------
    raw : list[dict]
        Records returned by ``fetch_division_species``.

    Returns
    -------
    pl.DataFrame
        One row per genome; columns: species, common_name, taxon_id,
        assembly, strain.
    """
    rows = []
    for rec in raw:
        rows.append({
            "species":      rec.get("name", ""),           # e.g. "arabidopsis_thaliana"
            "common_name":  rec.get("common_name", ""),    # e.g. "thale cress"
            "taxon_id":     rec.get("taxon_id", None),     # NCBI taxon integer
            "assembly":     rec.get("assembly_name", ""),  # e.g. "TAIR10"
            "strain":       rec.get("strain", ""),         # cultivar/strain name
        })
    return pl.DataFrame(rows).with_columns(
        pl.col("taxon_id").cast(pl.Int64, strict=False)
    )


df_plants = parse_division_species(plants_raw)

print(f"Shape  : {df_plants.shape}")
print(f"Dtypes : {df_plants.dtypes}")
df_plants.head(8)

### 1.6 Fetch EnsemblBacteria Species List (sample)

In [ ]:
BACTERIA_CACHE = DATA_DIR / "ensembl_bacteria_species.json"

# EnsemblBacteria has ~50,000 genomes; the /info/genomes/division endpoint
# returns them all in one call but the response is very large (~20 MB JSON).
# We fetch it once and cache; on re-runs the cache is loaded instantly.
bacteria_raw = fetch_division_species("EnsemblBacteria", BACTERIA_CACHE)
df_bacteria = parse_division_species(bacteria_raw)

print(f"EnsemblBacteria genomes : {len(bacteria_raw)}")
print(f"Shape  : {df_bacteria.shape}")
print(f"Dtypes : {df_bacteria.dtypes}")
df_bacteria.head(5)

### 1.7 Gene Lookup — AT1G01010 (*Arabidopsis thaliana*)

In [ ]:
# AT1G01010 encodes NAC domain-containing protein 1 (NAC001), the first
# annotated gene on Arabidopsis thaliana chromosome 1 and a standard
# reference locus used in genome-annotation benchmarks.

gene_info = eg_get("/lookup/symbol/arabidopsis_thaliana/AT1G01010")

print("=== AT1G01010 gene record ===")
print(f"  Gene ID      : {gene_info.get('id')}")
print(f"  Display name : {gene_info.get('display_name')}")
print(f"  Biotype      : {gene_info.get('biotype')}")
print(f"  Description  : {gene_info.get('description')}")
print(f"  Location     : {gene_info.get('seq_region_name')}:"
      f"{gene_info.get('start')}–{gene_info.get('end')} "
      f"(strand {gene_info.get('strand')})")
print(f"  Assembly     : {gene_info.get('assembly_name')}")
print(f"  Species      : {gene_info.get('species')}")

### 1.8 Cross-Division Summary DataFrame

In [ ]:
# We already have EnsemblPlants and EnsemblBacteria counts from the cached
# species lists.  For the remaining three divisions we query the API once each.

DIVISION_CACHES: dict[str, Path] = {
    "EnsemblPlants":    PLANTS_CACHE,
    "EnsemblBacteria":  BACTERIA_CACHE,
    "EnsemblFungi":     DATA_DIR / "ensembl_fungi_species.json",
    "EnsemblMetazoa":   DATA_DIR / "ensembl_metazoa_species.json",
    "EnsemblProtists":  DATA_DIR / "ensembl_protists_species.json",
}

division_counts: dict[str, int] = {}
for division, cache_path in DIVISION_CACHES.items():
    raw = fetch_division_species(division, cache_path)
    division_counts[division] = len(raw)

# Build a compact summary DataFrame — one row per division
df_divisions = pl.DataFrame({
    "division":     list(division_counts.keys()),
    "species_count": list(division_counts.values()),
}).sort("species_count", descending=True)

print("Cross-division species counts:")
print(df_divisions)
print(f"\nTotal genomes across all divisions: {df_divisions['species_count'].sum():,}")

### 1.9 Gene Counts for Model Organisms Across Divisions

In [ ]:
# Model organisms, each from a different Ensembl Genomes division
MODEL_ORGANISMS = [
    ("arabidopsis_thaliana",   "EnsemblPlants"),
    ("saccharomyces_cerevisiae", "EnsemblFungi"),
    ("drosophila_melanogaster", "EnsemblMetazoa"),
]

GENE_COUNTS_CACHE = DATA_DIR / "model_organism_gene_counts.json"


def fetch_gene_counts(species: str) -> dict:
    """Fetch gene-count statistics for one species from the Ensembl REST API.

    The ``/info/genomes/<species>`` endpoint returns a genome metadata record
    that includes ``gene_count`` (all annotated loci) and
    ``coding_gene_count`` (protein-coding genes only).

    Parameters
    ----------
    species : str
        Ensembl species name in lowercase-underscore format,
        e.g. ``"arabidopsis_thaliana"``.

    Returns
    -------
    dict
        Raw genome metadata record from the API.
    """
    return eg_get(f"/info/genomes/{species}")


if GENE_COUNTS_CACHE.exists():
    print(f"Loading cached gene counts from {GENE_COUNTS_CACHE}")
    gene_count_data = json.loads(GENE_COUNTS_CACHE.read_text())
else:
    gene_count_data = {}
    for species, division in MODEL_ORGANISMS:
        print(f"Fetching genome metadata for {species} ({division}) ...")
        rec = fetch_gene_counts(species)
        rec["_division"] = division   # preserve division label
        gene_count_data[species] = rec
    GENE_COUNTS_CACHE.write_text(json.dumps(gene_count_data, indent=2))
    print(f"Saved to {GENE_COUNTS_CACHE}")


# --- Parse into a Polars DataFrame ---
gc_rows = []
for species, rec in gene_count_data.items():
    gc_rows.append({
        "species":           species,
        "division":          rec.get("_division", ""),
        "gene_count":        rec.get("gene_count", None),
        "coding_gene_count": rec.get("coding_count", None),   # field name in API
        "assembly":          rec.get("assembly_name", ""),
    })

df_gene_counts = pl.DataFrame(gc_rows).with_columns(
    pl.col("gene_count").cast(pl.Int64, strict=False),
    pl.col("coding_gene_count").cast(pl.Int64, strict=False),
)

print(f"\nShape  : {df_gene_counts.shape}")
print(f"Dtypes :")
for col, dtype in zip(df_gene_counts.columns, df_gene_counts.dtypes):
    print(f"  {col:<22} {dtype}")
print()
print(df_gene_counts)